In [ ]:
# !pip install requests beautifulsoup4 pandas openpyxl lxml -q

import os
import re
import json
import time
import random
import traceback
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

# =============================
# CONFIG
# =============================
SITE_NAME = "yallatyre"
BASE_URL = "https://www.yallatyre.com"   # если редиректит на www/без www — оставь как в браузере

OUT_XLSX = f"{SITE_NAME}_tyres.xlsx"
CHECKPOINT_PATH = f"{SITE_NAME}_checkpoint.json"
URLCACHE_PATH = f"{SITE_NAME}_all_product_urls.json"

SAVE_EVERY_ROWS = 200        # автосейв каждые N новых строк
SLEEP = 0.35                 # базовая задержка между запросами
VERBOSE_ITEMS = True         # печатать OK на каждый товар
MAX_403_STREAK = 15          # если подряд 403 — сохранить прогресс и выйти
MAX_CRAWL_PAGES_FALLBACK = 5000  # лимит для fallback-crawl (если нет sitemap)
MAX_CRAWL_DEPTH_FALLBACK = 4     # глубина обхода для fallback-crawl
TEST_LIMIT = None            # None = без лимита

# =============================
# HTTP SESSION + RETRIES
# =============================
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9,ru;q=0.8",
})

def jitter_sleep(base=SLEEP, mult=1.0):
    time.sleep(base * mult + random.uniform(0, base))

def request_with_retries(url: str, tries: int = 6, timeout: int = 35) -> requests.Response:
    last = None
    for t in range(1, tries + 1):
        try:
            r = SESSION.get(url, timeout=timeout)
            return r
        except Exception as e:
            last = e
            wait = min(12, 1.2 ** t) + random.uniform(0, 1.5)
            print(f"NET ERR retry {t}/{tries} -> sleep {wait:.2f}s | {type(e).__name__}: {e}")
            time.sleep(wait)
    raise last

def get_soup(url: str) -> tuple[BeautifulSoup, int]:
    r = request_with_retries(url)
    status = r.status_code
    print(f"    GET {url} -> {status}")
    # не raise_for_status тут — 404/403 обрабатываем выше по логике
    jitter_sleep()
    return BeautifulSoup(r.text, "html.parser"), status

def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", (x or "")).strip()

def is_same_host(url: str) -> bool:
    try:
        host = urlparse(url).netloc.lower()
        base = urlparse(BASE_URL).netloc.lower()
        return host == base
    except:
        return False

# =============================
# SIZE NORMALIZATION: 195/65R15
# =============================
def normalize_size(size_text: str) -> str | None:
    t = clean_text(size_text)
    # 205/55 R16 -> 205/55R16
    m = re.search(r"(\d{3})\s*/\s*(\d{2,3})\s*R\s*(\d{2})", t, flags=re.I)
    if m:
        return f"{m.group(1)}/{m.group(2)}R{m.group(3)}"
    # 205-55-r16 -> 205/55R16
    m2 = re.search(r"(\d{3})[-/ ](\d{2,3})[-/ ]r(\d{2})", t, flags=re.I)
    if m2:
        return f"{m2.group(1)}/{m2.group(2)}R{m2.group(3)}"
    return None

# =============================
# DISCOVERY: SITEMAP
# =============================
def discover_sitemaps() -> list[str]:
    """
    Пытаемся найти sitemap:
      - /robots.txt (Sitemap:)
      - типовые пути: /sitemap.xml, /sitemap_index.xml, /sitemap-en.xml ...
    """
    candidates = []

    # 1) robots.txt
    robots_url = urljoin(BASE_URL, "/robots.txt")
    try:
        r = request_with_retries(robots_url, tries=3, timeout=25)
        if r.status_code == 200:
            for line in r.text.splitlines():
                if line.lower().startswith("sitemap:"):
                    sm = line.split(":", 1)[1].strip()
                    if sm:
                        candidates.append(sm)
    except Exception:
        pass

    # 2) common
    common = [
        "/sitemap.xml",
        "/sitemap_index.xml",
        "/sitemap-index.xml",
        "/sitemap-en.xml",
        "/sitemap1.xml",
        "/sitemap/sitemap.xml",
    ]
    for p in common:
        candidates.append(urljoin(BASE_URL, p))

    # ping each (быстро)
    uniq = []
    for u in candidates:
        if u not in uniq:
            uniq.append(u)

    ok = []
    for u in uniq:
        try:
            r = request_with_retries(u, tries=2, timeout=20)
            if r.status_code == 200 and ("<urlset" in r.text or "<sitemapindex" in r.text):
                ok.append(u)
        except Exception:
            pass

    ok = list(dict.fromkeys(ok))
    return ok

def parse_sitemap_xml(xml_text: str) -> tuple[list[str], list[str]]:
    """
    Возвращает (urls, sitemaps)
    """
    urls = []
    sitemaps = []
    # sitemapindex
    for loc in re.findall(r"<loc>(.*?)</loc>", xml_text, flags=re.I | re.S):
        loc = clean_text(loc)
        if not loc:
            continue
        sitemaps.append(loc)

    # если это urlset — loc те же, но мы не знаем тип по regex
    # различим по наличию <urlset
    if "<urlset" in xml_text.lower():
        urls = sitemaps
        sitemaps = []

    return urls, sitemaps

def collect_all_sitemap_urls(roots: list[str], limit_sitemaps: int = 2000) -> list[str]:
    all_urls = []
    queue = list(roots)
    seen_sm = set()

    while queue and len(seen_sm) < limit_sitemaps:
        sm = queue.pop(0)
        if sm in seen_sm:
            continue
        seen_sm.add(sm)

        try:
            r = request_with_retries(sm, tries=3, timeout=30)
            if r.status_code != 200:
                continue
            txt = r.text
            urls, nested = parse_sitemap_xml(txt)
            if urls:
                all_urls.extend(urls)
            if nested:
                for n in nested:
                    if n not in seen_sm:
                        queue.append(n)
        except Exception:
            continue

    # normalize
    out = []
    for u in all_urls:
        u = clean_text(u)
        if u and u.startswith("http") and is_same_host(u):
            out.append(u)

    return list(dict.fromkeys(out))

# =============================
# DISCOVERY: FALLBACK CRAWL (если нет sitemap)
# =============================
def is_probably_product_url(u: str) -> bool:
    """
    Эвристика "похоже на карточку товара".
    Под yallatyre может понадобиться подстройка — но эта версия обычно хорошо стартует.
    """
    path = urlparse(u).path.lower()

    bad = ["/cart", "/checkout", "/account", "/login", "/register", "/wishlist", "/compare", "/search", "/contact"]
    if any(b in path for b in bad):
        return False

    # частые паттерны товара
    good_markers = ["/product", "/tyre", "/tire", "/item", "/shop", "/shop/"]
    if any(g in path for g in good_markers):
        # исключим списки/категории (часто короткие пути)
        if path.count("/") >= 2:
            return True

    # иногда товар — длинный slug без /product
    # heuristics: содержит размерность
    if re.search(r"\b\d{3}[-/ ]\d{2,3}[-/ ]r\d{2}\b", path, flags=re.I):
        return True

    return False

def fallback_crawl_product_urls(start_url: str | None = None) -> list[str]:
    """
    BFS по сайту, собираем ссылки, выделяем product_like.
    """
    start = start_url or BASE_URL
    q = [(start, 0)]
    seen_pages = set()
    product_urls = set()

    while q and len(seen_pages) < MAX_CRAWL_PAGES_FALLBACK:
        url, depth = q.pop(0)
        if url in seen_pages:
            continue
        seen_pages.add(url)

        soup, status = get_soup(url)
        if status != 200:
            continue

        # собираем ссылки
        links = set()
        for a in soup.select("a[href]"):
            href = a.get("href", "").strip()
            if not href:
                continue
            absu = urljoin(url, href)
            # чистим якоря
            absu = absu.split("#", 1)[0]
            if not absu.startswith("http"):
                continue
            if not is_same_host(absu):
                continue
            links.add(absu)

        for u in links:
            if is_probably_product_url(u):
                product_urls.add(u)
            # углубляемся
            if depth + 1 <= MAX_CRAWL_DEPTH_FALLBACK and u not in seen_pages:
                q.append((u, depth + 1))

        if len(seen_pages) % 200 == 0:
            print(f"--- fallback crawl pages={len(seen_pages)} | product_like={len(product_urls)} ---")

    return sorted(product_urls)

# =============================
# URL CACHE
# =============================
def load_or_build_url_cache() -> list[str]:
    if os.path.exists(URLCACHE_PATH):
        with open(URLCACHE_PATH, "r", encoding="utf-8") as f:
            j = json.load(f)
        urls = j.get("product_urls", []) or []
        urls = [u for u in urls if isinstance(u, str) and u.startswith("http")]
        print(f"✅ URLs cache loaded: {URLCACHE_PATH} | product_urls={len(urls)}")
        return urls

    roots = discover_sitemaps()
    if roots:
        print("✅ Found sitemap candidates:")
        for r in roots[:10]:
            print("   -", r)
        all_urls = collect_all_sitemap_urls(roots)
        product_like = [u for u in all_urls if is_probably_product_url(u)]
        product_like = list(dict.fromkeys(product_like))

        with open(URLCACHE_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "base_url": BASE_URL,
                "total_raw": len(all_urls),
                "product_like": len(product_like),
                "product_urls": product_like,
                "built_at": time.ctime(),
                "method": "sitemap",
                "roots": roots,
            }, f, ensure_ascii=False, indent=2)

        print(f"✅ URLs cached -> {URLCACHE_PATH} | total_raw={len(all_urls)} | product_like={len(product_like)}")
        return product_like

    print("⚠️ No sitemap found. Using fallback crawl (may be slower / may miss hidden URLs).")
    product_urls = fallback_crawl_product_urls()
    with open(URLCACHE_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "base_url": BASE_URL,
            "product_urls": product_urls,
            "built_at": time.ctime(),
            "method": "fallback_crawl",
            "pages_limit": MAX_CRAWL_PAGES_FALLBACK,
            "depth": MAX_CRAWL_DEPTH_FALLBACK,
        }, f, ensure_ascii=False, indent=2)

    print(f"✅ URLs cached -> {URLCACHE_PATH} | product_urls={len(product_urls)}")
    return product_urls

# =============================
# CHECKPOINT
# =============================
def load_checkpoint() -> dict:
    if not os.path.exists(CHECKPOINT_PATH):
        return {"next_index": 0, "rows": 0, "mode": "parse"}
    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {"next_index": 0, "rows": 0, "mode": "parse"}

def save_checkpoint(next_index: int, rows: int, mode: str = "parse"):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "next_index": int(next_index),
            "rows": int(rows),
            "mode": mode,
            "saved_at": time.ctime(),
        }, f, ensure_ascii=False, indent=2)
    print(f"✅ Checkpoint -> {os.path.abspath(CHECKPOINT_PATH)} | next_index={next_index} | rows={rows} | mode={mode}")

# =============================
# EXCEL SAVE + PIVOT
# =============================
def build_pivot_min_price(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()

    needed = {"size", "brand", "price"}
    if not needed.issubset(df_raw.columns):
        return pd.DataFrame()

    df = df_raw.copy()
    df["Price"] = pd.to_numeric(df["price"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df["Size"] = df["size"].astype(str).str.strip()
    df["Brand"] = df["brand"].astype(str).str.strip()

    df = df.dropna(subset=["Price"]).copy()
    df = df[(df["Size"] != "") & (df["Brand"] != "")]

    pivot = df.pivot_table(index="Size", columns="Brand", values="Price", aggfunc="min").round(2)
    return pivot.sort_index()

def save_all(out_path: str, rows: list[dict]):
    df = pd.DataFrame(rows)
    pivot = build_pivot_min_price(df)

    tmp_path = out_path + ".__tmp__.xlsx"
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as w:
        df.to_excel(w, index=False, sheet_name="Data")
        pivot.to_excel(w, sheet_name="Pivot")

    os.replace(tmp_path, out_path)

    st = os.stat(out_path)
    uniq = df["url"].nunique() if ("url" in df.columns and len(df) > 0) else 0
    print(f"💾 SAVE -> {os.path.abspath(out_path)} | rows={len(df)} | uniq_url={uniq} | mtime={time.ctime(st.st_mtime)}")

def load_existing(out_path: str) -> tuple[list[dict], set[str]]:
    if not os.path.exists(out_path):
        return [], set()

    try:
        df = pd.read_excel(out_path, sheet_name="Data")
        if df is None or df.empty or "url" not in df.columns:
            return [], set()

        df = df.dropna(subset=["url"]).copy()
        df["url"] = df["url"].astype(str).str.strip()
        seen = set(df["url"].tolist())

        rows = df.to_dict(orient="records")
        print(f"📌 Existing file loaded: {os.path.basename(out_path)} | rows={len(rows)} | seen_urls={len(seen)}")
        return rows, seen
    except Exception as e:
        print(f"⚠️ Could not read existing file {out_path}: {e}")
        return [], set()

# =============================
# PRODUCT PAGE PARSING (only if price exists)
# =============================
def extract_jsonld_product(soup: BeautifulSoup) -> dict | None:
    """
    Ищем JSON-LD schema.org Product
    """
    scripts = soup.find_all("script", attrs={"type": "application/ld+json"})
    for sc in scripts:
        txt = sc.get_text(strip=True)
        if not txt:
            continue
        try:
            data = json.loads(txt)
        except Exception:
            continue

        # может быть list или dict
        candidates = data if isinstance(data, list) else [data]
        for obj in candidates:
            if not isinstance(obj, dict):
                continue
            t = obj.get("@type")
            if isinstance(t, list):
                ok = any(str(x).lower() == "product" for x in t)
            else:
                ok = str(t).lower() == "product"
            if ok:
                return obj
    return None

def parse_price_from_text(text: str) -> Decimal | None:
    t = clean_text(text)
    # вытаскиваем число с точкой/запятой
    m = re.search(r"(\d{1,3}(?:[,\s]\d{3})*(?:\.\d+)?|\d+(?:\.\d+)?)", t)
    if not m:
        return None
    raw = m.group(1).replace(" ", "").replace(",", "")
    try:
        d = Decimal(raw)
        return d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    except InvalidOperation:
        return None

def parse_product_page(url: str) -> dict | None:
    soup, status = get_soup(url)

    if status == 404:
        return None
    if status == 403:
        # специальная обработка выше по циклу (через streak)
        raise PermissionError("403 Forbidden")

    if status != 200:
        return None

    # 1) JSON-LD product (если есть — самый надёжный)
    brand = None
    model = None
    price = None
    currency = None

    jprod = extract_jsonld_product(soup)
    if jprod:
        model = clean_text(str(jprod.get("name") or "")) or None
        b = jprod.get("brand")
        if isinstance(b, dict):
            brand = clean_text(str(b.get("name") or "")) or None
        elif isinstance(b, str):
            brand = clean_text(b) or None

        offers = jprod.get("offers")
        # offers может быть dict или list
        offer_list = offers if isinstance(offers, list) else ([offers] if isinstance(offers, dict) else [])
        for off in offer_list:
            if not isinstance(off, dict):
                continue
            p = off.get("price")
            cur = off.get("priceCurrency")
            if p is not None:
                try:
                    d = Decimal(str(p))
                    price = d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
                    currency = str(cur) if cur else None
                    break
                except InvalidOperation:
                    pass

    # 2) fallback: заголовок страницы
    if not model:
        h1 = soup.select_one("h1")
        if h1:
            model = clean_text(h1.get_text(" ", strip=True)) or None
    if not brand and model:
        # очень грубо: бренд часто первым словом в title/h1
        maybe = model.split(" ")[0]
        if maybe and len(maybe) >= 3:
            brand = maybe

    # 3) price: fallback по DOM (типовые селекторы магазинов)
    if price is None:
        candidates = [
            ".price .amount",
            ".price",
            ".product-price",
            ".woocommerce-Price-amount",
            "ins .amount",
            ".summary .price",
        ]
        for sel in candidates:
            node = soup.select_one(sel)
            if node:
                p = parse_price_from_text(node.get_text(" ", strip=True))
                if p is not None:
                    price = p
                    break

    # если цены нет — "не актуальный товар" для твоей задачи
    if price is None:
        return None

    # 4) size: из model/h1/url по regex
    size = None
    if model:
        size = normalize_size(model)
    if not size:
        size = normalize_size(urlparse(url).path)
    if not size:
        # возможно есть атрибуты/таблица
        txt = soup.get_text(" ", strip=True)
        size = normalize_size(txt)

    # 5) year (опционально, если найдём 20xx рядом)
    year = None
    txt = soup.get_text(" ", strip=True)
    m = re.search(r"\b(20\d{2})\b", txt)
    if m:
        y = int(m.group(1))
        if 2000 <= y <= 2035:
            year = y

    row = {
        "size": size or "",
        "brand": brand or "UNKNOWN",
        "model": model or "",
        "year": year,
        "price": str(price),
        "currency": currency or "",
        "url": url,
    }
    return row

# =============================
# MAIN
# =============================
print(f"\n=== {SITE_NAME.upper()} PARSER (pitstop-style: sitemap/cache/checkpoint/seen_urls) ===")
print("CWD:", os.getcwd())
print("OUT:", os.path.abspath(OUT_XLSX))
print("CHK:", os.path.abspath(CHECKPOINT_PATH))
print("URLCACHE:", os.path.abspath(URLCACHE_PATH))

# 1) urls cache
product_urls = load_or_build_url_cache()
print("\nВсего URL товаров к обработке:", len(product_urls))

# 2) load existing rows + seen
all_rows, seen_urls = load_existing(OUT_XLSX)

# 3) checkpoint
cp = load_checkpoint()
next_index = int(cp.get("next_index", 0) or 0)
if next_index < 0:
    next_index = 0
if next_index > len(product_urls):
    next_index = 0

print(f"Resume: next_index={next_index} | already_rows={len(all_rows)} | seen_urls={len(seen_urls)}\n")

last_saved_rows = len(all_rows)
added_since_save = 0
streak_403 = 0

def add_row_if_new(row: dict) -> bool:
    global added_since_save, last_saved_rows
    url = (row.get("url") or "").strip()
    if not url or url in seen_urls:
        return False
    seen_urls.add(url)
    all_rows.append(row)
    return True

skipped_404 = 0
skipped_no_price = 0
skipped_other = 0

for idx in range(next_index, len(product_urls)):
    url = product_urls[idx]

    # прогресс-строка
    if idx % 50 == 0:
        print(f"--- progress: idx={idx}/{len(product_urls)} | rows={len(all_rows)} | uniq={len(seen_urls)} ---")

    # лимит для тестов
    if TEST_LIMIT is not None and len(all_rows) >= TEST_LIMIT:
        print("⏹️ TEST_LIMIT reached.")
        next_index = idx
        break

    try:
        row = parse_product_page(url)

        # 404/не товар/нет цены
        if row is None:
            # если 404 — parse_product_page вернул None, но статус внутри не отличаем
            # грубо классифицируем по окончанию url/паттернам не будем — просто no_price
            skipped_no_price += 1
        else:
            streak_403 = 0  # сброс серии 403

            added = add_row_if_new(row)
            if added:
                added_since_save += 1
                if VERBOSE_ITEMS:
                    print(f"    OK: {row.get('brand')} | {row.get('size')} | {row.get('model')} | year={row.get('year')} | price={row.get('price')} | {url}")

        # autosave
        if added_since_save >= SAVE_EVERY_ROWS:
            save_all(OUT_XLSX, all_rows)
            save_checkpoint(next_index=idx + 1, rows=len(all_rows), mode="parse")
            added_since_save = 0

    except PermissionError:
        streak_403 += 1
        print(f"    403 streak: {streak_403}/{MAX_403_STREAK} -> {url}")
        if streak_403 >= MAX_403_STREAK:
            print("\n⚠️ Too many 403 подряд. Похоже, VPN/доступ сломался.")
            print("⚠️ Сохраняю прогресс и выхожу. Включи VPN и запусти ячейку снова — продолжит с этого места.\n")
            save_all(OUT_XLSX, all_rows)
            save_checkpoint(next_index=idx, rows=len(all_rows), mode="parse")
            break

    except requests.HTTPError as e:
        # если где-то всё же raise_for_status сработает
        msg = str(e)
        if "404" in msg:
            skipped_404 += 1
        else:
            skipped_other += 1
        print(f"❌ HTTPError on {url}: {e}")
        save_all(OUT_XLSX, all_rows)
        save_checkpoint(next_index=idx, rows=len(all_rows), mode="parse")
        break

    except Exception as e:
        skipped_other += 1
        print(f"\n❌ Ошибка на URL: {url}\n   {type(e).__name__}: {e}")
        # при любой критической ошибке — сохранить и выйти
        save_all(OUT_XLSX, all_rows)
        save_checkpoint(next_index=idx, rows=len(all_rows), mode="parse")
        print("Stacktrace (short):")
        traceback.print_exc(limit=2)
        break

else:
    # если цикл прошёл полностью
    save_all(OUT_XLSX, all_rows)
    save_checkpoint(next_index=len(product_urls), rows=len(all_rows), mode="parse")

print("\n========================================")
df_final = pd.DataFrame(all_rows)
uniq = df_final["url"].nunique() if ("url" in df_final.columns and len(df_final) > 0) else 0
print(f"✅ DONE. rows={len(df_final)} | uniq_url={uniq}")
print(f"skipped approx: no_price_or_not_product={skipped_no_price} | other={skipped_other}")
print(f"✅ File: {os.path.abspath(OUT_XLSX)} (sheets: Data, Pivot)")
print(f"✅ Checkpoint: {os.path.abspath(CHECKPOINT_PATH)}")
print(f"✅ URL cache: {os.path.abspath(URLCACHE_PATH)}")


Выходные данные были обрезаны до нескольких последних строк (5000).
    OK: HABILEAD | 205/60R16 | Habilead 205/60 R16 92V Comfort Max H202 2025 | year=2025 | price=177.00 | https://www.yallatyre.com/habilead-205-60-r16-92v-comfort-max-h202-2025.html
    GET https://www.yallatyre.com/habilead-215-60-r16-95v-comfort-max-h202-2025.html -> 200
    OK: HABILEAD | 215/60R16 | Habilead 215/60 R16 95V Comfort Max H202 2025 | year=2025 | price=171.00 | https://www.yallatyre.com/habilead-215-60-r16-95v-comfort-max-h202-2025.html
    GET https://www.yallatyre.com/habilead-275-50-r20-113w-practicalmax-h-p-rs26-2025.html -> 200
    OK: HABILEAD | 275/50R20 | Habilead 275/50 R20 113W Practicalmax H/P RS26 2025 | year=2025 | price=309.00 | https://www.yallatyre.com/habilead-275-50-r20-113w-practicalmax-h-p-rs26-2025.html
    GET https://www.yallatyre.com/habilead-165-65-r14-79h-comfort-max-h202-2025.html -> 200
    OK: HABILEAD | 165/65R14 | Habilead 165/65 R14 79H Comfort Max H202 2025 | year=2025 

In [ ]:
# ==========================================
# YALLATYRE PARSER (pitstop-style)
# sitemap/robots -> URL cache -> checkpoint -> seen_urls
# Save to Google Drive (safe across Colab restarts)
# ==========================================

# !pip -q install requests beautifulsoup4 pandas openpyxl lxml

import os
import re
import json
import time
import random
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

# ---------- 0) Mount Google Drive ----------
from google.colab import drive
drive.mount("/content/drive")

# ---------- 1) CONFIG ----------
BASE_URL = "https://www.yallatyre.com"
DOMAIN = urlparse(BASE_URL).netloc.lower()

# куда сохраняем (Drive)
OUT_XLSX = "/content/drive/MyDrive/yallatyre_tyres.xlsx"
CHECKPOINT_PATH = "/content/drive/MyDrive/yallatyre_checkpoint.json"
URLCACHE_PATH = "/content/drive/MyDrive/yallatyre_all_product_urls.json"

# скорость/поведение
SLEEP = 0.25
TIMEOUT = 40
SAVE_EVERY_ROWS = 200          # автосейв каждые N НОВЫХ уникальных строк
VERBOSE_ITEMS = True           # печатать OK для каждого товара
MAX_DISCOVERY_URLS = None      # None = без лимита (на всякий)
MAX_CRAWL_PAGES = 20000        # запасной план (если sitemap нет)
MAX_RETRIES = 6

# если нужно ограничить для теста:
TEST_LIMIT = None              # None = без лимита (по уникальным товарам)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Connection": "keep-alive",
})

# ---------- 2) Helpers ----------
def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", (x or "")).strip()

def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def request_with_retries(url: str):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = SESSION.get(url, timeout=TIMEOUT)
            # 403/429/5xx часто временные — ретраим
            if r.status_code in (403, 429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}", response=r)
            return r
        except Exception as e:
            last_err = e
            sleep_s = min(20, (1.2 ** attempt) + random.random() * 1.2)
            print(f"NET ERR retry {attempt}/{MAX_RETRIES} -> sleep {sleep_s:.2f}s | {type(e).__name__}: {e}")
            time.sleep(sleep_s)
    raise last_err

def get_soup(url: str, sleep: float = 0.0) -> BeautifulSoup:
    r = request_with_retries(url)
    print(f"    GET {url} -> {r.status_code}")
    r.raise_for_status()
    if sleep:
        time.sleep(sleep)
    return BeautifulSoup(r.text, "lxml")

def is_same_domain(url: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(DOMAIN)
    except Exception:
        return False

def normalize_size_to_195_65R15(text: str) -> str | None:
    """
    Приводим к виду: 195/65R15
    На yallatyre часто размер зашит в заголовке/URL.
    """
    t = clean_text(text).replace("-", " ")
    m = re.search(r"\b(\d{3})\s*/?\s*(\d{2,3})\s*[Rr]\s*(\d{2})\b", t)
    if m:
        return f"{m.group(1)}/{m.group(2)}R{m.group(3)}"
    return None

def extract_year_any(text: str) -> int | None:
    if not text:
        return None
    yrs = re.findall(r"\b(20\d{2})\b", text)
    if yrs:
        try:
            return int(yrs[-1])
        except:
            return None
    return None

def extract_price_from_jsonld(soup: BeautifulSoup) -> Decimal | None:
    """
    Пытаемся вытащить цену из JSON-LD.
    """
    scripts = soup.select('script[type="application/ld+json"]')
    for sc in scripts:
        raw = sc.get_text("\n", strip=True)
        if not raw:
            continue
        # JSON-LD бывает массивом или объектом
        candidates = []
        try:
            data = json.loads(raw)
            candidates = data if isinstance(data, list) else [data]
        except Exception:
            # иногда там несколько JSON подряд — попробуем грубо
            continue

        for obj in candidates:
            if not isinstance(obj, dict):
                continue
            # Product / Offer
            offers = obj.get("offers")
            if isinstance(offers, dict):
                offers = [offers]
            if isinstance(offers, list):
                for off in offers:
                    if not isinstance(off, dict):
                        continue
                    p = off.get("price")
                    if p is None:
                        continue
                    try:
                        d = Decimal(str(p)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
                        return d
                    except InvalidOperation:
                        pass
    return None

def extract_price_fallback(soup: BeautifulSoup) -> Decimal | None:
    """
    Фолбэк: itemprop=price / meta / текст с AED / regex по странице.
    """
    # itemprop
    node = soup.select_one('[itemprop="price"]')
    if node:
        val = node.get("content") or node.get_text(" ", strip=True)
        val = clean_text(val)
        m = re.search(r"(\d+(?:\.\d+)?)", val.replace(",", ""))
        if m:
            try:
                return Decimal(m.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
            except InvalidOperation:
                pass

    # meta price
    for sel in ['meta[property="product:price:amount"]', 'meta[name="price"]', 'meta[property="og:price:amount"]']:
        mnode = soup.select_one(sel)
        if mnode and mnode.get("content"):
            val = mnode["content"].strip()
            m = re.search(r"(\d+(?:\.\d+)?)", val.replace(",", ""))
            if m:
                try:
                    return Decimal(m.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
                except InvalidOperation:
                    pass

    # грубый regex по тексту рядом с AED
    txt = soup.get_text(" ", strip=True)
    txt = txt.replace(",", "")
    m = re.search(r"\bAED\s*(\d+(?:\.\d+)?)\b", txt, flags=re.I)
    if m:
        try:
            return Decimal(m.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        except InvalidOperation:
            pass

    # просто первая "разумная" цена (осторожно)
    m = re.search(r"\b(\d{2,5}(?:\.\d{1,2})?)\b", txt)
    if m:
        try:
            d = Decimal(m.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
            # отсечём совсем маленькие (например, номера)
            if d >= Decimal("10"):
                return d
        except InvalidOperation:
            pass
    return None

def extract_title(soup: BeautifulSoup) -> str | None:
    h1 = soup.select_one("h1")
    if h1:
        t = clean_text(h1.get_text(" ", strip=True))
        if t:
            return t
    og = soup.select_one('meta[property="og:title"]')
    if og and og.get("content"):
        return clean_text(og["content"])
    if soup.title:
        t = clean_text(soup.title.get_text(" ", strip=True))
        if t:
            return t
    return None

def guess_brand_from_title(title: str) -> str | None:
    if not title:
        return None
    # бренд обычно первая "словарная" часть
    first = re.split(r"[\|\-–—/]", title)[0]
    first = clean_text(first)
    # если в начале размер — тогда бренд не угадать
    if normalize_size_to_195_65R15(first):
        return None
    tok = first.split()
    if tok:
        return tok[0].strip().upper()
    return None

def parse_product_page(url: str) -> dict | None:
    soup = get_soup(url, sleep=SLEEP)

    title = extract_title(soup)
    if not title:
        return None

    # быстрый фильтр: если это явно блог/контент — выходим
    if "/blog" in urlparse(url).path.lower():
        return None

    # цена
    price = extract_price_from_jsonld(soup)
    if price is None:
        price = extract_price_fallback(soup)

    # если цены нет — считаем, что неактуально/не товар (по твоему требованию "только актуальные с ценой")
    if price is None:
        return None

    # размер / год / бренд
    size = normalize_size_to_195_65R15(title) or normalize_size_to_195_65R15(url)
    year = extract_year_any(title) or extract_year_any(url)

    brand = guess_brand_from_title(title) or "UNKNOWN"
    model = clean_text(title)

    return {
        "size": size,
        "brand": brand,
        "model": model,
        "year": year,
        "price": str(price),
        "url": url
    }

# ---------- 3) Sitemap / robots discovery ----------
def discover_sitemaps() -> list[str]:
    """
    Ищем sitemap через robots.txt и стандартные пути.
    Возвращаем список URL sitemap-ов.
    """
    candidates = [
        urljoin(BASE_URL, "/sitemap.xml"),
        urljoin(BASE_URL, "/sitemap_index.xml"),
        urljoin(BASE_URL, "/sitemap-index.xml"),
        urljoin(BASE_URL, "/sitemap.php"),
        urljoin(BASE_URL, "/sitemap/sitemap.xml"),
    ]

    # robots.txt
    robots = urljoin(BASE_URL, "/robots.txt")
    try:
        r = request_with_retries(robots)
        if r.status_code == 200:
            for line in r.text.splitlines():
                if line.lower().startswith("sitemap:"):
                    sm = line.split(":", 1)[1].strip()
                    if sm:
                        candidates.append(sm)
    except Exception:
        pass

    # uniq keep order
    seen = set()
    out = []
    for c in candidates:
        if not c:
            continue
        if c in seen:
            continue
        seen.add(c)
        out.append(c)

    # проверим какие реально существуют (xml)
    good = []
    for u in out:
        try:
            r = request_with_retries(u)
            if r.status_code == 200 and ("<urlset" in r.text or "<sitemapindex" in r.text):
                good.append(u)
        except Exception:
            pass

    return good

def parse_sitemap_xml(xml_text: str) -> tuple[list[str], list[str]]:
    """
    Возвращает (urls, sitemap_links)
    """
    urls = re.findall(r"<loc>\s*([^<\s]+)\s*</loc>", xml_text)
    # если это sitemapindex — там тоже <loc>, но мы отдельно определим по тегу
    if "<sitemapindex" in xml_text:
        return [], urls
    return urls, []

def collect_all_sitemap_urls(sitemaps: list[str]) -> list[str]:
    """
    Обходит sitemapindex -> urlset (в несколько уровней).
    """
    queue = list(sitemaps)
    visited = set()
    all_urls = []

    while queue:
        sm = queue.pop(0)
        if sm in visited:
            continue
        visited.add(sm)

        r = request_with_retries(sm)
        if r.status_code != 200:
            continue
        txt = r.text

        urls, submaps = parse_sitemap_xml(txt)
        if submaps:
            for s in submaps:
                if s not in visited:
                    queue.append(s)
        else:
            all_urls.extend(urls)

        if MAX_DISCOVERY_URLS is not None and len(all_urls) >= MAX_DISCOVERY_URLS:
            all_urls = all_urls[:MAX_DISCOVERY_URLS]
            break

    # uniq keep order
    seen = set()
    out = []
    for u in all_urls:
        if u in seen:
            continue
        seen.add(u)
        out.append(u)
    return out

# ---------- 4) Fallback crawl (if no sitemap) ----------
def looks_like_product_url(u: str) -> bool:
    if not u:
        return False
    if not is_same_domain(u):
        return False
    p = urlparse(u).path.lower()

    # выкидываем очевидный мусор
    if any(x in p for x in ["/blog", "/category", "/tag", "/cart", "/checkout", "/account", "/privacy", "/terms"]):
        return False

    # yallatyre: много товаров в *.html
    if p.endswith(".html"):
        return True

    # иногда /en/product/... (на всякий)
    if "/product/" in p:
        return True

    return False

def crawl_collect_product_urls(start_url: str) -> list[str]:
    """
    BFS crawl по сайту, собирает product-like URL.
    Запасной план, если sitemap нет.
    """
    queue = [start_url]
    seen_pages = set()
    product_urls = []
    seen_product = set()

    while queue and len(seen_pages) < MAX_CRAWL_PAGES:
        u = queue.pop(0)
        if u in seen_pages:
            continue
        seen_pages.add(u)

        try:
            soup = get_soup(u, sleep=SLEEP)
        except Exception as e:
            print(f"crawl skip {u} | {type(e).__name__}: {e}")
            continue

        # все ссылки со страницы
        for a in soup.select("a[href]"):
            href = a.get("href", "").strip()
            if not href:
                continue
            nu = urljoin(u, href)
            nu = nu.split("#")[0]

            if not is_same_domain(nu):
                continue

            if looks_like_product_url(nu):
                if nu not in seen_product:
                    seen_product.add(nu)
                    product_urls.append(nu)

            # добавляем в очередь "контентные" страницы, чтобы дойти до товаров
            # но не добавляем сами товары (их мы и так собираем)
            if (not looks_like_product_url(nu)) and (nu not in seen_pages):
                # не раздуваем очередь мусором
                if any(x in urlparse(nu).path.lower() for x in [".jpg", ".png", ".webp", ".css", ".js", ".pdf"]):
                    continue
                queue.append(nu)

        if MAX_DISCOVERY_URLS is not None and len(product_urls) >= MAX_DISCOVERY_URLS:
            product_urls = product_urls[:MAX_DISCOVERY_URLS]
            break

    return product_urls

# ---------- 5) URL Cache ----------
def load_or_build_url_cache() -> list[str]:
    if os.path.exists(URLCACHE_PATH):
        try:
            with open(URLCACHE_PATH, "r", encoding="utf-8") as f:
                data = json.load(f)
            urls = data.get("product_urls") if isinstance(data, dict) else data
            urls = [u for u in urls if isinstance(u, str)]
            print(f"✅ URLs cache loaded: {os.path.basename(URLCACHE_PATH)} | product_urls={len(urls)}")
            return urls
        except Exception as e:
            print(f"⚠️ Не смог прочитать URL cache, пересоберу: {e}")

    # 1) sitemap
    roots = discover_sitemaps()
    if roots:
        all_urls = collect_all_sitemap_urls(roots)
        product_urls = [u for u in all_urls if looks_like_product_url(u)]
        # uniq keep order
        seen = set()
        product_urls2 = []
        for u in product_urls:
            if u in seen:
                continue
            seen.add(u)
            product_urls2.append(u)
        product_urls = product_urls2
        print(f"✅ Sitemap discovery: sitemaps={len(roots)} | total_raw={len(all_urls)} | product_like={len(product_urls)}")
    else:
        # 2) fallback crawl
        print("⚠️ Sitemap не найден — включаю fallback crawl (может быть дольше).")
        product_urls = crawl_collect_product_urls(BASE_URL)
        print(f"✅ Crawl discovery: product_urls={len(product_urls)}")

    with open(URLCACHE_PATH, "w", encoding="utf-8") as f:
        json.dump({"base_url": BASE_URL, "ts": now_ts(), "product_urls": product_urls}, f, ensure_ascii=False, indent=2)

    print(f"✅ URLs cached -> {URLCACHE_PATH} | product_urls={len(product_urls)}")
    return product_urls

# ---------- 6) Checkpoint ----------
def load_checkpoint() -> dict | None:
    if not os.path.exists(CHECKPOINT_PATH):
        return None
    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def save_checkpoint(next_index: int, rows_count: int, mode: str = "parse"):
    payload = {
        "base_url": BASE_URL,
        "ts": now_ts(),
        "mode": mode,
        "next_index": int(next_index),
        "rows_count": int(rows_count),
    }
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"✅ Checkpoint -> {CHECKPOINT_PATH} | next_index={next_index} | rows={rows_count} | mode={mode}")

# ---------- 7) Excel save / load existing ----------
def build_pivot_min_price(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    needed = {"size", "brand", "price"}
    if not needed.issubset(df_raw.columns):
        return pd.DataFrame()

    df = df_raw.copy()
    df["Price"] = pd.to_numeric(df["price"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df["Size"] = df["size"].astype(str).str.strip()
    df["Brand"] = df["brand"].astype(str).str.strip()
    df = df.dropna(subset=["Price"]).copy()
    df = df[(df["Size"] != "") & (df["Brand"] != "")]
    pivot = df.pivot_table(index="Size", columns="Brand", values="Price", aggfunc="min").round(2)
    return pivot.sort_index()

def save_all(rows: list[dict]):
    df = pd.DataFrame(rows)
    pivot = build_pivot_min_price(df)

    tmp_path = OUT_XLSX + ".__tmp__.xlsx"
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as w:
        df.to_excel(w, index=False, sheet_name="Data")
        pivot.to_excel(w, sheet_name="Pivot")

    os.replace(tmp_path, OUT_XLSX)
    st = os.stat(OUT_XLSX)
    uniq = df["url"].nunique() if ("url" in df.columns and len(df) > 0) else 0
    print(f"💾 SAVE -> {OUT_XLSX} | rows={len(df)} | uniq_url={uniq} | mtime={time.ctime(st.st_mtime)}")

def load_existing() -> tuple[list[dict], set[str]]:
    if not os.path.exists(OUT_XLSX):
        return [], set()
    try:
        df = pd.read_excel(OUT_XLSX, sheet_name="Data")
        if df is None or df.empty or "url" not in df.columns:
            return [], set()
        df = df.dropna(subset=["url"]).copy()
        df["url"] = df["url"].astype(str).str.strip()
        seen = set(df["url"].tolist())
        rows = df.to_dict(orient="records")
        print(f"📌 Existing file loaded: {os.path.basename(OUT_XLSX)} | rows={len(rows)} | seen_urls={len(seen)}")
        return rows, seen
    except Exception as e:
        print(f"⚠️ Не смог прочитать существующий OUT_XLSX: {e}")
        return [], set()

# ---------- 8) MAIN ----------
print("\n=== YALLATYRE PARSER (pitstop-style: sitemap/cache/checkpoint/seen_urls) ===")
print("CWD:", os.getcwd())
print("OUT:", OUT_XLSX)
print("CHK:", CHECKPOINT_PATH)
print("URLCACHE:", URLCACHE_PATH)

# 1) urls cache
product_urls = load_or_build_url_cache()
print("\nВсего URL товаров к обработке:", len(product_urls))

# 2) load existing rows + seen
all_rows, seen_urls = load_existing()

# 3) resume from checkpoint
cp = load_checkpoint()
next_index = 0
if cp and isinstance(cp, dict) and "next_index" in cp:
    try:
        next_index = int(cp["next_index"])
    except:
        next_index = 0

# safety
if next_index < 0:
    next_index = 0
if next_index > len(product_urls):
    next_index = len(product_urls)

print(f"Resume: next_index={next_index} | already_rows={len(all_rows)} | seen_urls={len(seen_urls)}")

added_since_save = 0
processed = 0
skipped_no_price_or_not_product = 0
skipped_other = 0

def progress_line(i: int):
    uniq = len(seen_urls)
    print(f"--- progress: idx={i}/{len(product_urls)} | processed={processed} | added_since_save={added_since_save} | uniq={uniq} ---")

try:
    for i in range(next_index, len(product_urls)):
        url = product_urls[i]

        # если уже есть — пропускаем
        if url in seen_urls:
            processed += 1
            continue

        # парсим товар
        try:
            row = parse_product_page(url)
        except requests.HTTPError as he:
            # 404/прочее — пропуск, но продолжаем; чекпоинт сохраним
            print(f"\n❌ HTTPError on URL: {url}\n   {he}")
            skipped_other += 1
            processed += 1
            if (processed % 200) == 0:
                progress_line(i)
            save_checkpoint(next_index=i+1, rows_count=len(all_rows), mode="parse")
            continue
        except Exception as e:
            print(f"\n❌ Error on URL: {url}\n   {type(e).__name__}: {e}")
            skipped_other += 1
            processed += 1
            if (processed % 200) == 0:
                progress_line(i)
            save_checkpoint(next_index=i+1, rows_count=len(all_rows), mode="parse")
            continue

        processed += 1

        if row is None:
            skipped_no_price_or_not_product += 1
        else:
            u = (row.get("url") or "").strip()
            if u and (u not in seen_urls):
                seen_urls.add(u)
                all_rows.append(row)
                added_since_save += 1

                if VERBOSE_ITEMS:
                    print(f"OK: {row.get('brand')} | {row.get('size')} | {row.get('model')} | year={row.get('year')} | price={row.get('price')} | {u}")

                if TEST_LIMIT is not None and len(all_rows) >= TEST_LIMIT:
                    print("⏹️ TEST_LIMIT reached, stopping.")
                    save_all(all_rows)
                    save_checkpoint(next_index=i+1, rows_count=len(all_rows), mode="parse")
                    break

        # автосейв
        if added_since_save >= SAVE_EVERY_ROWS:
            save_all(all_rows)
            save_checkpoint(next_index=i+1, rows_count=len(all_rows), mode="parse")
            added_since_save = 0

        # прогресс (раз в ~200 обработанных)
        if (processed % 200) == 0:
            progress_line(i)

    # финальный сейв
    save_all(all_rows)
    save_checkpoint(next_index=len(product_urls), rows_count=len(all_rows), mode="parse")

except KeyboardInterrupt:
    print("\n⛔ Interrupted by user. Saving progress...")
    save_all(all_rows)
    # i может не существовать если упали раньше — страхуем
    try:
        save_checkpoint(next_index=i, rows_count=len(all_rows), mode="parse")
    except Exception:
        save_checkpoint(next_index=next_index, rows_count=len(all_rows), mode="parse")

print("\n========================================")
df_final = pd.DataFrame(all_rows)
uniq = df_final["url"].nunique() if ("url" in df_final.columns and len(df_final) > 0) else 0
print(f"✅ DONE. rows={len(df_final)} | uniq_url={uniq}")
print(f"skipped approx: no_price_or_not_product={skipped_no_price_or_not_product} | other={skipped_other}")
print("✅ File:", OUT_XLSX, "(sheets: Data, Pivot)")
print("✅ Checkpoint:", CHECKPOINT_PATH)
print("✅ URL cache:", URLCACHE_PATH)


Выходные данные были обрезаны до нескольких последних строк (5000).
    GET https://www.yallatyre.com/habilead-225-65-r17-102h-k717-2025.html -> 200
OK: HABILEAD | 225/65R17 | Habilead 225/65 R17 102H K717 2025 | year=2025 | price=210.00 | https://www.yallatyre.com/habilead-225-65-r17-102h-k717-2025.html
    GET https://www.yallatyre.com/habilead-205-45-r16-87v-comfort-max-h202-2025.html -> 200
OK: HABILEAD | 205/45R16 | Habilead 205/45 R16 87V Comfort Max H202 2025 | year=2025 | price=145.00 | https://www.yallatyre.com/habilead-205-45-r16-87v-comfort-max-h202-2025.html
    GET https://www.yallatyre.com/habilead-175-65-r14-82h-comfort-max-h202-2025.html -> 200
OK: HABILEAD | 175/65R14 | Habilead 175/65 R14 82H Comfort Max H202 2025 | year=2025 | price=125.00 | https://www.yallatyre.com/habilead-175-65-r14-82h-comfort-max-h202-2025.html
    GET https://www.yallatyre.com/habilead-225-70-r16-103t-comfort-max-h202-2025.html -> 200
OK: HABILEAD | 225/70R16 | Habilead 225/70 R16 103T Comfort